# Path Patching Results
Load saved tensors from `results/` and compare the 2B and 12B heatmaps.

In [ ]:
from pathlib import Path

import torch
import plotly.express as px
from plotly.subplots import make_subplots

ROOT = Path('results')
models = [
    ('gemma_2b', ROOT / 'gemma_2b' / 'path_patch_final_resid.pt'),
    ('gemma_12b', ROOT / 'gemma_12b' / 'path_patch_final_resid.pt'),
]

fig = make_subplots(rows=1, cols=2, subplot_titles=[name for name, _ in models])

for col, (name, path) in enumerate(models, start=1):
    tensor = torch.load(path, map_location='cpu')
    heatmap_fig = px.imshow(
        tensor,
        color_continuous_scale='RdBu',
        zmin=-float(tensor.abs().max()),
        zmax=float(tensor.abs().max()),
        origin='lower',
        labels={'x': 'Head', 'y': 'Layer', 'color': 'Factual recall'},
        title=f'{name} path patching',
    )
    trace = heatmap_fig.data[0]
    trace.coloraxis = 'coloraxis'
    fig.add_trace(trace, row=1, col=col)

fig.update_layout(
    coloraxis=dict(colorscale='RdBu', cmin=-1, cmid=0, cmax=1),
    height=500,
    width=1200,
    title='Path Patching Comparison',
)
fig.update_xaxes(title_text='Head')
fig.update_yaxes(title_text='Layer')
fig.show()